# Prompt 1 — audited potential graph
The original 01_build_graphs.ipynb is preserved unchanged. This canonical-name notebook executes the current notebook-owned implementation. MultiDiGraph is selected to preserve parallel candidates even if this dataset has no parallel pairs. Semantic mappings remain unverified; dataset type coverage is verified separately. Probability/variance belong to the later ML phase, not Prompt 2.

In [1]:
from pathlib import Path
import json, sys, hashlib, statistics
from collections import Counter, defaultdict
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'data_model/dataset/attack_path_dataset/data/model_input').is_dir())
sys.path.insert(0,str(ROOT))
DATA = ROOT/'data_model/dataset/attack_path_dataset/data'
MODEL = DATA/'model_input'
AUDIT = ROOT/'phase_a_audit'
ART = AUDIT/'artifacts'
ART.mkdir(parents=True,exist_ok=True)
TWIN_SOURCE_SHA = 'acfc662a54d5fcf02bd7adc896897946e6b35eae19cc500314000633f6c113f5'
checks = []
def check(name, condition):
    checks.append({'name':name,'passed':bool(condition)})
    print(('PASS: ' if condition else 'FAIL: ')+name)
    assert condition, name
def current_inputs():
    return {p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(MODEL.glob('*')) if p.is_file()}
def save_stage(stage, payload):
    envelope = dict(run_id=run_id, input_hashes=current_inputs(), source_sha=TWIN_SOURCE_SHA, payload=payload)
    (ART/(stage+'.json')).write_text(json.dumps(envelope),encoding='utf-8')
def load_stage(stage):
    envelope = json.loads((ART/(stage+'.json')).read_text(encoding='utf-8'))
    check(stage+' current input fingerprints',envelope['input_hashes']==current_inputs())
    check(stage+' current source fingerprint',envelope['source_sha']==TWIN_SOURCE_SHA)
    check(stage+' current execution run',envelope['run_id']==run_id)
    return envelope['payload']
def report(stage, values):
    values['checks'] = checks.copy()
    (AUDIT/(stage+'_metrics.json')).write_text(json.dumps(values,indent=2),encoding='utf-8')
    print(json.dumps(values,indent=2))
def graph_pack(graphs):
    return {sid:dict(metadata=g.graph,nodes=[[n,d] for n,d in g.nodes(data=True)],edges=[[u,v,k,d] for u,v,k,d in g.edges(keys=True,data=True)]) for sid,g in graphs.items()}
def graph_unpack(packed):
    import networkx as nx
    graphs={}
    for sid,entry in packed.items():
        g=nx.MultiDiGraph()
        g.graph.update(entry['metadata'])
        g.add_nodes_from((n,d) for n,d in entry['nodes'])
        for u,v,k,d in entry['edges']:
            g.add_edge(u,v,key=k,**d)
        graphs[sid]=g
    return graphs

run_id = json.loads((ART/'00_recon.json').read_text())['run_id']


In [2]:
"""
twin.py  —  AttackTwin AI: Digital Twin pipeline functions
============================================================
SOURCE OF TRUTH: notebooks/03_observed_layer_ledger.ipynb and
                 notebooks/04_twin_export_inference.ipynb.
This file is generated by %%writefile in Prompt 4; do NOT edit it manually.
Import this module from the API or from other notebooks:
    from twin import (normalise_event, resolve_entity, resolve_events, run_dedup,
                      build_potential_graph, build_ledger, compute_coverage,
                      build_observed_layer, export_graph_json)
"""
from __future__ import annotations
import json, os, re, math
from collections import defaultdict, Counter
from datetime import datetime, timezone
from typing import Any, Optional
from pydantic import BaseModel, Field

# ────────────────────────────────────────────────────────────────────────────
# SECTION 1 — Mapping tables (VERIFIED against live data, 2026-10-14)
# ────────────────────────────────────────────────────────────────────────────
ASSET_PLAN_TYPE = {
    "user_identity":           "identity",
    "privileged_identity":     "identity",  # is_admin=True flagged separately
    "cloud_identity":          "identity",
    "endpoint":                "host",
    "server":                  "host",
    "external_entry":          "host",      # perimeter ingress
    "application":             "service",
    "database":                "data",
    "security_infrastructure": "control",
}

REL_PLAN_TYPE = {
    "NETWORK_REACHABILITY":     "network_flow",
    "AUTHENTICATES_TO":         "auth",
    "CAN_ASSUME_IDENTITY":      "privilege",
    "HAS_PRIVILEGE":            "privilege",
    "CAN_ACCESS_DATA":          "data_access",
    "DEPLOYS_TO":               "deploy",
    "TRUSTS_SERVICE":           "trust",
    "ADMINISTERS":              "admin",
    "CAN_MODIFY_CONFIGURATION": "config_change",
}


# ────────────────────────────────────────────────────────────────────────────
# SECTION 2 — Normaliser
# Per-source rules extracted from generate_dataset.py (encode_sev / fmt_ts)
# and confirmed against raw events and normalized_events.jsonl.
# ────────────────────────────────────────────────────────────────────────────

def _parse_ts(source: str, raw) -> datetime:
    """Return aware UTC datetime from source-specific raw timestamp."""
    if source == "synthetic_netflow":
        return datetime.fromtimestamp(int(raw), tz=timezone.utc)
    if source == "synthetic_api_gateway":
        return datetime.fromtimestamp(int(raw) // 1000, tz=timezone.utc)
    if source == "synthetic_edr":
        return datetime.strptime(raw, "%Y-%m-%d %H:%M:%S").replace(tzinfo=timezone.utc)
    return datetime.strptime(raw, "%Y-%m-%dT%H:%M:%SZ").replace(tzinfo=timezone.utc)


def _decode_sev(source: str, raw) -> str:
    """Map source-specific raw severity to canonical low|medium|high|critical."""
    if raw is None:
        return "low"
    if source == "synthetic_idp_logs":
        return {"info": "low", "warn": "medium", "alert": "high", "critical": "critical"}.get(raw, "low")
    if source == "synthetic_cloud_audit":
        return str(raw).lower()
    if source == "synthetic_api_gateway":
        return {"notice": "low", "warning": "medium", "error": "high", "critical": "critical"}.get(raw, "low")
    if source == "synthetic_secret_scan":
        return str(raw).lower()
    if source == "synthetic_edr":
        r = int(raw)
        return "low" if r <= 3 else "medium" if r <= 6 else "high" if r <= 8 else "critical"
    if source == "synthetic_vuln_scan":
        r = float(raw)
        return "low" if r < 4 else "medium" if r < 7 else "high" if r < 9 else "critical"
    return "low"


def normalise_event(raw_event: dict, evidence_row: dict) -> dict:
    """
    Produce the common event schema from raw events.jsonl row + evidence.jsonl row.
    """
    src = raw_event["source"]
    dt  = _parse_ts(src, raw_event["timestamp"])
    ts  = dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    sev = _decode_sev(src, raw_event.get("severity_raw"))
    return {
        "event_id":        raw_event["event_id"],
        "ts":              ts,
        "source":          src,
        "event_type":      raw_event["event_type"],
        "actor":           raw_event.get("source_entity_raw"),
        "target":          raw_event.get("target_entity_raw"),
        "severity":        sev,
        "confidence":      evidence_row["collection_confidence"],
        "observed_fields": raw_event.get("observed_fields", {}),
        "scenario_id":     raw_event["scenario_id"],
    }


# ────────────────────────────────────────────────────────────────────────────
# SECTION 3 — Alias Resolver (Soft, Reversible)
# ────────────────────────────────────────────────────────────────────────────

def build_alias_index(assets: list[dict], scenario_id: str) -> dict:
    """
    Build per-scenario lookup maps from every known alias form -> asset_id.
    """
    idx: dict[str, dict[str, list[str]]] = {
        "exact_id":       {},
        "exact_email":    {},
        "exact_hostname": {},
        "exact_ip":       {},
        "ci_id":          {},
        "ci_email":       {},
        "ci_hostname":    {},
        "ci_ip":          {},
        "display_name":   {},
    }
    scen_assets = [a for a in assets if a["scenario_id"] == scenario_id]
    for a in scen_assets:
        aid = a["asset_id"]
        def _add(bucket, key):
            if key:
                idx[bucket].setdefault(key, []).append(aid)
                idx["ci_" + bucket.split("_", 1)[1]].setdefault(key.lower(), []).append(aid)
        _add("exact_id",       aid)
        _add("exact_email",    a.get("email", "") or "")
        _add("exact_hostname", a.get("hostname", "") or "")
        _add("exact_ip",       a.get("ip_address", "") or "")
        dn = a.get("display_name") or ""
        if dn:
            idx["display_name"].setdefault(dn.lower(), []).append(aid)
    return idx


def resolve_entity(
    ref: str | None,
    alias_index: dict,
    scenario_id: str,
) -> dict:
    """
    Softly resolve one entity reference.
    """
    if not ref:
        return {"ref": ref, "resolved": [], "confidence": "unresolved",
                "method": "null_ref", "ambiguous": False}
    ref_l = ref.lower()

    if ref in alias_index["exact_id"]:
        return {"ref": ref, "resolved": [ref], "confidence": "high",
                "method": "exact_id", "ambiguous": False}
    if ref in alias_index["exact_email"]:
        cands = alias_index["exact_email"][ref]
        return {"ref": ref, "resolved": cands, "confidence": "low" if len(cands) > 1 else "high",
                "method": "exact_email", "ambiguous": len(cands) > 1}
    if ref in alias_index["exact_hostname"]:
        cands = alias_index["exact_hostname"][ref]
        return {"ref": ref, "resolved": cands, "confidence": "low" if len(cands) > 1 else "high",
                "method": "exact_hostname", "ambiguous": len(cands) > 1}
    if ref in alias_index["exact_ip"]:
        cands = alias_index["exact_ip"][ref]
        return {"ref": ref, "resolved": cands, "confidence": "low" if len(cands) > 1 else "high",
                "method": "exact_ip", "ambiguous": len(cands) > 1}
    if ref_l in alias_index["ci_id"]:
        cands = alias_index["ci_id"][ref_l]
        return {"ref": ref, "resolved": cands, "confidence": "low" if len(cands) > 1 else "high",
                "method": "ci_id", "ambiguous": len(cands) > 1}
    if ref_l in alias_index["ci_email"]:
        cands = alias_index["ci_email"][ref_l]
        return {"ref": ref, "resolved": cands, "confidence": "low" if len(cands) > 1 else "high",
                "method": "ci_email", "ambiguous": len(cands) > 1}
    if ref_l in alias_index["ci_hostname"]:
        cands = alias_index["ci_hostname"][ref_l]
        return {"ref": ref, "resolved": cands, "confidence": "low" if len(cands) > 1 else "high",
                "method": "ci_hostname", "ambiguous": len(cands) > 1}
    if ref_l in alias_index["ci_ip"]:
        cands = alias_index["ci_ip"][ref_l]
        return {"ref": ref, "resolved": cands, "confidence": "low" if len(cands) > 1 else "high",
                "method": "ci_ip", "ambiguous": len(cands) > 1}
    if ref_l in alias_index["display_name"]:
        cands = alias_index["display_name"][ref_l]
        if len(cands) == 1:
            return {"ref": ref, "resolved": cands, "confidence": "medium",
                    "method": "display_name_unique", "ambiguous": False}
        else:
            return {"ref": ref, "resolved": cands, "confidence": "low",
                    "method": "display_name_ambiguous", "ambiguous": True}

    return {"ref": ref, "resolved": [], "confidence": "unresolved",
            "method": "no_match", "ambiguous": False}


def resolve_events(
    norm_events: list[dict],
    assets: list[dict],
) -> tuple[list[dict], list[dict]]:
    """
    Adds actor_resolved and target_resolved to each normalised event.
    Returns (norm_events, merges_log).
    """
    scenario_ids = list({e["scenario_id"] for e in norm_events})
    alias_indices = {
        sid: build_alias_index(assets, sid) for sid in scenario_ids
    }
    merges_log = []
    for ev in norm_events:
        idx = alias_indices[ev["scenario_id"]]
        for role, raw in [("actor", ev["actor"]), ("target", ev["target"])]:
            res = resolve_entity(raw, idx, ev["scenario_id"])
            ev[f"{role}_resolved"] = res
            if res["confidence"] != "unresolved":
                merges_log.append({
                    "event_id":    ev["event_id"],
                    "scenario_id": ev["scenario_id"],
                    "role":        role,
                    "raw_ref":     raw,
                    "resolved":    res["resolved"],
                    "confidence":  res["confidence"],
                    "method":      res["method"],
                    "ambiguous":   res["ambiguous"],
                    "reversible":  True,
                })
    return norm_events, merges_log


# ────────────────────────────────────────────────────────────────────────────
# SECTION 4 — Deduplicator (Near-duplicate detector)
# ────────────────────────────────────────────────────────────────────────────

def run_dedup(norm_events: list[dict]) -> list[dict]:
    """
    Flag near-duplicate events in-place.
    Matches across same (scenario, event_type) and overlapping resolved entity pair
    within |ts_delta| <= 20 seconds.
    """
    def get_ids(ev, role):
        res = ev.get(f"{role}_resolved")
        if res and res.get("resolved"):
            return frozenset(res["resolved"])
        raw = ev.get(role)
        return frozenset([raw]) if raw else frozenset()

    for ev in norm_events:
        ev.pop("duplicate_of", None)
    groups = defaultdict(list)
    for i, ev in enumerate(norm_events):
        key = (ev["scenario_id"], ev["event_type"])
        dt = datetime.strptime(ev["ts"], "%Y-%m-%dT%H:%M:%SZ").replace(tzinfo=timezone.utc)
        groups[key].append((dt, i))

    for entries in groups.values():
        if len(entries) < 2:
            continue
        entries.sort(key=lambda x: (x[0], x[1]))
        for j in range(1, len(entries)):
            dt_j, idx_j = entries[j]
            ev_j = norm_events[idx_j]
            if ev_j.get("duplicate_of") is not None:
                continue
            actor_j = get_ids(ev_j, "actor")
            target_j = get_ids(ev_j, "target")
            for k in range(j - 1, -1, -1):
                dt_k, idx_k = entries[k]
                delta = (dt_j - dt_k).total_seconds()
                if delta > 20:
                    break
                ev_k = norm_events[idx_k]
                if ev_k.get("duplicate_of") is not None:
                    continue
                actor_k = get_ids(ev_k, "actor")
                target_k = get_ids(ev_k, "target")
                if bool(actor_j & actor_k) and bool(target_j & target_k):
                    ev_j["duplicate_of"] = ev_k["event_id"]
                    break

    return norm_events


# ────────────────────────────────────────────────────────────────────────────
# SECTION 5 — Graph Builder (Prompt 1 Potential Graph)
# ────────────────────────────────────────────────────────────────────────────

try:
    import networkx as nx
    _HAS_NX = True
except ImportError:
    _HAS_NX = False


def build_potential_graph(
    scenario_id: str,
    assets: list[dict],
    relationships: list[dict],
    scenario_row: dict,
    cost_actions_by_edge: dict | None = None,
) -> "nx.DiGraph":
    if not _HAS_NX:
        raise ImportError("networkx is required")
    G = nx.MultiDiGraph()
    ca = cost_actions_by_edge or {}
    for a in assets:
        if a["scenario_id"] != scenario_id:
            continue
        at = a["asset_type"]
        G.add_node(
            a["asset_id"],
            plan_type                = ASSET_PLAN_TYPE[at],
            dataset_type             = at,
            layer                    = "potential",
            zone                     = a["zone"],
            exposure                 = a["exposure"],
            criticality              = a["criticality"],
            n_controls               = len(a.get("security_controls", [])),
            identity_confidence      = a["identity_confidence"],
            configuration_confidence = a["configuration_confidence"],
            evidence_coverage        = a["evidence_coverage"],
            is_admin                 = (at == "privileged_identity"),
            display_name             = a.get("display_name"),
            email                    = a.get("email"),
            hostname                 = a.get("hostname"),
            ip_address               = a.get("ip_address"),
        )
    for r in relationships:
        if r["scenario_id"] != scenario_id:
            continue
        rt  = r["relationship_type"]
        eid = r["edge_id"]
        G.add_edge(
            r["source_id"], r["target_id"], key=eid,
            edge_id           = eid,
            plan_type         = REL_PLAN_TYPE[rt],
            dataset_type      = rt,
            layer             = "potential",
            evidence_basis    = r["evidence_basis"],
            prior_probability = r["prior_probability"],
            p                 = None,
            variance          = None,
            evidence          = [],
            cost_actions      = list(ca.get((scenario_id, eid), [])),
        )
    shared_groups = {}
    for node, data in G.nodes(data=True):
        if data["plan_type"] == "identity":
            out_eids = [d["edge_id"] for _, _, d in G.out_edges(node, data=True)]
            if out_eids:
                shared_groups[node] = out_eids
    G.graph.update({
        "scenario_id":      scenario_id,
        "entry_points":     scenario_row["entry_points"],
        "critical_targets": scenario_row["critical_targets"],
        "shared_groups":    shared_groups,
    })
    return G



import logging
from pathlib import Path
from typing import Literal
from pydantic import ConfigDict, model_validator

_normalise_unchecked = normalise_event
KNOWN_EVENT_TYPES = {'policy_changed', 'privilege_change', 'connection_observed', 'malware_detected', 'database_read', 'health_check', 'access_blocked', 'vulnerability_detected', 'sensitive_resource_access', 'port_exposure_change', 'login_failure', 'mfa_challenge', 'connection_denied', 'role_assigned', 'unusual_request_rate', 'suspicious_process', 'login_success', 'authentication_anomaly', 'token_issued', 'service_exposed', 'acl_updated', 'service_error', 'policy_enforced', 'api_access', 'admin_login', 'secret_exposure_detected'}

def normalise_event(raw_event, evidence_row, errors=None):
    """Reject malformed records, retaining an audit entry; null entities are valid."""
    try:
        for field in ('event_id', 'scenario_id', 'source', 'timestamp', 'event_type'):
            if not isinstance(raw_event[field], (str, int, float)) or not raw_event[field]:
                raise ValueError('missing/invalid ' + field)
        if raw_event['event_type'] not in KNOWN_EVENT_TYPES:
            raise ValueError('unknown event type')
        if raw_event['source'] not in {'synthetic_netflow', 'synthetic_api_gateway', 'synthetic_edr', 'synthetic_idp_logs', 'synthetic_cloud_audit', 'synthetic_secret_scan', 'synthetic_vuln_scan'}:
            raise ValueError('unknown sensor')
        for field in ('source_entity_raw', 'target_entity_raw'):
            if raw_event.get(field) is not None and not isinstance(raw_event[field], str):
                raise ValueError('invalid entity reference')
        if not isinstance(raw_event.get('observed_fields', {}), dict):
            raise ValueError('invalid observed fields')
        if not 0 <= float(evidence_row['collection_confidence']) <= 1:
            raise ValueError('invalid confidence')
        return _normalise_unchecked(raw_event, evidence_row)
    except (KeyError, TypeError, ValueError, OverflowError, OSError) as exc:
        error = {'event_id': raw_event.get('event_id'), 'reason': str(exc)}
        if errors is not None:
            errors.append(error)
        logging.warning('Skipped telemetry: %s', error)
        return None

def build_ledger(norm_events, graphs):
    ledger = []
    weights = {'high': 1., 'medium': .7, 'low': .3, 'unresolved': 0.}
    for ev in norm_events:
        G = graphs[ev['scenario_id']]
        actor, target = ev['actor_resolved'], ev['target_resolved']
        ev['link_notes'] = []
        if not actor['resolved'] or not target['resolved']:
            ev['link_notes'].append('unresolved endpoint; no candidate edge; weight=0')
        if actor['confidence'] != 'high' or target['confidence'] != 'high':
            ev['link_notes'].append('identity uncertainty lowers candidate weight')
        for a in actor['resolved']:
            for b in target['resolved']:
                candidates = [d for d in G.get_edge_data(a, b, default={}).values() if d['layer']=='potential']
                for d in candidates:
                    ambiguous = len(candidates)>1 or actor['ambiguous'] or target['ambiguous']
                    weight = weights[actor['confidence']] * weights[target['confidence']] * ev['confidence']
                    if ambiguous:
                        weight *= .5
                    if ev.get('duplicate_of'):
                        weight = 0.
                    ledger.append(dict(scenario_id=ev['scenario_id'], event_id=ev['event_id'], edge_id=d['edge_id'], link='pair_match_ambiguous' if ambiguous else 'pair_match', weight_hint=weight, duplicate_of=ev.get('duplicate_of'), actor_confidence=actor['confidence'], target_confidence=target['confidence']))
                    if ev['event_id'] not in d['evidence']:
                        d['evidence'].append(ev['event_id'])
    return ledger

def build_observed_layer(graphs, norm_events):
    for ev in norm_events:
        G = graphs[ev['scenario_id']]
        eid = ev['event_id']
        node = '__event__' + eid
        G.add_node(node, layer='observed', plan_type='Event', dataset_type=ev['event_type'], event=ev.copy(), evidence=[eid])
        # Scanner fields describe a finding but do not establish a unique affected asset.
        # Secret location is a category, not an asset id. Keep only Event in these cases.
        target = ev['target_resolved']
        scanner = ev['event_type'] in ('secret_exposure_detected', 'vulnerability_detected')
        if ev['event_type']=='secret_exposure_detected' or (scanner and (len(target['resolved'])!=1 or target['confidence']!='high')):
            G.nodes[node]['assessment'] = '(c) affected asset unclear; Event only'
            continue
        def link(source, dest, kind):
            key = '__obs__' + eid + ':' + kind + ':' + dest
            G.add_edge(source, dest, key=key, edge_id=key, layer='observed', plan_type=kind, dataset_type=kind, evidence=[eid], prior_probability=None, p=None, variance=None, cost_actions=[])
        roles = ('target_resolved',) if scanner else ('actor_resolved', 'target_resolved')
        for role in roles:
            res = ev[role]
            if len(res['resolved'])==1 and res['confidence'] in ('high', 'medium'):
                link(node, res['resolved'][0], 'AFFECTED_BY')
        if ev['event_type']=='vulnerability_detected':
            finding = '__vuln__' + eid
            G.add_node(finding, layer='observed', plan_type='Vulnerability', dataset_type='vulnerability_detected', observed_fields=ev['observed_fields'], evidence=[eid], assessment='(b) target_entity_raw is affected asset; detection is not exploitation')
            link(finding, target['resolved'][0], 'EXPLOITS')

def compute_coverage(graphs, norm_events, ledger):
    events = {(e['scenario_id'], e['event_id']): e for e in norm_events}
    entity = defaultdict(set)
    for ev in norm_events:
        for role in ('actor_resolved', 'target_resolved'):
            for aid in ev[role]['resolved']:
                entity[(ev['scenario_id'], aid)].add(ev['event_id'])
    def coverage(sid, ids, noisy):
        unique = {eid for eid in ids if (sid,eid) in events and not events[(sid,eid)].get('duplicate_of')}
        return dict(n_events=len(unique), source_diversity=len({events[(sid,eid)]['source'] for eid in unique}), evidence_coverage=noisy, blind_spot=not unique)
    for sid, G in graphs.items():
        for nid, d in G.nodes(data=True):
            d['coverage'] = coverage(sid, entity[(sid,nid)] | set(d.get('evidence', [])), d.get('evidence_coverage'))
        for _, _, d in G.edges(data=True):
            d['coverage'] = coverage(sid, d.get('evidence', []), d.get('evidence_coverage'))
        G.graph['ledger'] = [r for r in ledger if r['scenario_id']==sid]

def read_rows(path):
    with Path(path).open(encoding='utf-8') as stream:
        return [json.loads(line) for line in stream if line.strip()]

def run_pipeline(model_input, raw_events=None):
    """Read model_input only. All intermediates are returned for notebook inspection."""
    path = Path(model_input)
    assets = read_rows(path/'assets.jsonl')
    relationships = read_rows(path/'relationships.jsonl')
    scenarios = read_rows(path/'scenarios.jsonl')
    evidence = {(r['scenario_id'], r['event_id']): r for r in read_rows(path/'evidence.jsonl')}
    actions = defaultdict(list)
    for row in read_rows(path/'interventions.jsonl'):
        for eid in row['affected_edge_ids']:
            actions[(row['scenario_id'],eid)].append(row['intervention_id'])
    graphs = {s['scenario_id']: build_potential_graph(s['scenario_id'], assets, relationships, s, actions) for s in scenarios}
    errors, events = [], []
    for raw in raw_events if raw_events is not None else read_rows(path/'events.jsonl'):
        ev = normalise_event(raw, evidence.get((raw.get('scenario_id'),raw.get('event_id')), {}), errors)
        if ev is not None:
            events.append(ev)
    events, merges = resolve_events(events, assets)
    run_dedup(events)
    ledger = build_ledger(events, graphs)
    build_observed_layer(graphs, events)
    compute_coverage(graphs, events, ledger)
    for sid,G in graphs.items():
        G.graph['merges'] = [m for m in merges if m['scenario_id']==sid]
    return graphs, events, ledger, merges, errors

class StrictModel(BaseModel):
    model_config = ConfigDict(extra='forbid')

class NodeModel(StrictModel):
    id: str
    plan_type: str
    dataset_type: str
    layer: Literal['potential', 'observed']
    attrs: dict[str, Any]
    aliases: list[str]
    coverage: dict[str, Any]

class EdgeModel(StrictModel):
    edge_id: str
    source: str
    target: str
    plan_type: str
    dataset_type: str
    layer: Literal['potential', 'observed']
    evidence_basis: str | None = None
    prior_probability: float | None = Field(default=None, ge=0, le=1)
    p: float | None = Field(default=None, ge=0, le=1)
    variance: float | None = Field(default=None, ge=0)
    evidence: list[str]
    cost_actions: list[str]
    coverage: dict[str, Any]

class LedgerRowModel(StrictModel):
    scenario_id: str
    event_id: str
    edge_id: str
    link: Literal['pair_match', 'pair_match_ambiguous']
    weight_hint: float = Field(ge=0, le=1)
    duplicate_of: str | None = None
    actor_confidence: str
    target_confidence: str

class MergeRowModel(StrictModel):
    alias: str
    asset_id: str
    confidence: float = Field(ge=0, le=1)
    reversible: Literal[True]
    event_id: str
    scenario_id: str
    role: str
    method: str
    ambiguous: bool

class ScenarioGraphModel(StrictModel):
    scenario_id: str
    nodes: list[NodeModel]
    edges: list[EdgeModel]
    entry_points: list[str]
    critical_targets: list[str]
    ledger: list[LedgerRowModel]
    merges: list[MergeRowModel]
    coverage: dict[str, Any]
    shared_groups: dict[str, list[str]]
    temporal_warnings: list[dict]

    @model_validator(mode='after')
    def references(self):
        nodes = {n.id for n in self.nodes}
        edges = {e.edge_id for e in self.edges}
        if len(nodes)!=len(self.nodes) or len(edges)!=len(self.edges):
            raise ValueError('duplicate scoped ids')
        if any(e.source not in nodes or e.target not in nodes for e in self.edges):
            raise ValueError('dangling endpoint')
        potential = {e.edge_id for e in self.edges if e.layer=='potential'}
        if any(r.edge_id not in potential or r.scenario_id!=self.scenario_id for r in self.ledger):
            raise ValueError('invalid ledger scope')
        if not set(self.entry_points+self.critical_targets) <= nodes:
            raise ValueError('unknown entry/target')
        return self

def export_graph_json(G, ledger=None, merges=None):
    sid = G.graph['scenario_id']
    nodes, edges = [], []
    for nid,d in G.nodes(data=True):
        attrs = {k:v for k,v in d.items() if k not in ('plan_type','dataset_type','layer','coverage')}
        aliases = [d[k] for k in ('display_name','email','hostname','ip_address') if d.get(k)]
        nodes.append(dict(id=nid, plan_type=d['plan_type'], dataset_type=d['dataset_type'], layer=d['layer'], attrs=attrs, aliases=aliases, coverage=d['coverage']))
    for u,v,d in G.edges(data=True):
        edges.append(dict(edge_id=d['edge_id'], source=u, target=v, **{k:d.get(k) for k in ('plan_type','dataset_type','layer','evidence_basis','prior_probability','p','variance','evidence','cost_actions','coverage')}))
    merge_rows = []
    for m in merges if merges is not None else G.graph.get('merges', []):
        if m['scenario_id']==sid:
            for aid in m['resolved']:
                merge_rows.append(dict(alias=m['raw_ref'], asset_id=aid, confidence={'high':1.,'medium':.7,'low':.3}[m['confidence']], **{k:m[k] for k in ('event_id','scenario_id','role','method','ambiguous','reversible')}))
    doc = dict(scenario_id=sid, nodes=nodes, edges=edges, ledger=[r for r in (ledger if ledger is not None else G.graph.get('ledger', [])) if r['scenario_id']==sid], merges=merge_rows, entry_points=G.graph['entry_points'], critical_targets=G.graph['critical_targets'], shared_groups=G.graph['shared_groups'], coverage={n['id']:n['coverage'] for n in nodes}, temporal_warnings=G.graph.get('temporal_warnings', []))
    return ScenarioGraphModel.model_validate(doc).model_dump()


In [3]:
recon=load_stage('00_recon')
tables=recon['tables']
assets,relationships,scenarios,interventions=(tables[k] for k in ('assets','relationships','scenarios','interventions'))
actions=defaultdict(list)
for it in interventions:
    for eid in it['affected_edge_ids']:actions[(it['scenario_id'],eid)].append(it['intervention_id'])
missing_assets=sorted({a['asset_type'] for a in assets}-ASSET_PLAN_TYPE.keys())
missing_edges=sorted({r['relationship_type'] for r in relationships}-REL_PLAN_TYPE.keys())
check('Every dataset type has a mapping',not missing_assets and not missing_edges)
graphs={s['scenario_id']:build_potential_graph(s['scenario_id'],assets,relationships,s,actions) for s in scenarios}
expected_nodes={(a['scenario_id'],a['asset_id'],a['asset_type']) for a in assets}
actual_nodes={(sid,n,d['dataset_type']) for sid,g in graphs.items() for n,d in g.nodes(data=True)}
expected_edges={(r['scenario_id'],r['edge_id'],r['source_id'],r['target_id'],r['relationship_type']) for r in relationships}
actual_edges={(sid,d['edge_id'],u,v,d['dataset_type']) for sid,g in graphs.items() for u,v,d in g.edges(data=True)}
check('All original node IDs and dataset types preserved',expected_nodes==actual_nodes)
check('All original edge IDs, endpoints and dataset types preserved',expected_edges==actual_edges)
check('Node counts equal input rows',sum(g.number_of_nodes() for g in graphs.values())==len(assets))
check('Edge counts equal input rows',sum(g.number_of_edges() for g in graphs.values())==len(relationships))
check('Entry points and critical targets exist',all(set(g.graph['entry_points']+g.graph['critical_targets'])<=set(g.nodes) for g in graphs.values()))
check('Every endpoint refers to an input asset',all((sid,u) in {(a['scenario_id'],a['asset_id']) for a in assets} and (sid,v) in {(a['scenario_id'],a['asset_id']) for a in assets} for sid,g in graphs.items() for u,v in g.edges()))
pairs=Counter((r['scenario_id'],r['source_id'],r['target_id']) for r in relationships)
per_scenario={sid:dict(nodes=g.number_of_nodes(),edges=g.number_of_edges(),entry_points=g.graph['entry_points'],critical_targets=g.graph['critical_targets'],shared_groups=g.graph['shared_groups'],intervention_linked_edges=sum(bool(d['cost_actions']) for _,_,d in g.edges(data=True))) for sid,g in graphs.items()}
metrics=dict(scenarios=len(graphs),nodes=sum(g.number_of_nodes() for g in graphs.values()),edges=sum(g.number_of_edges() for g in graphs.values()),per_scenario=per_scenario,node_plan_types=dict(Counter(d['plan_type'] for g in graphs.values() for _,d in g.nodes(data=True))),edge_plan_types=dict(Counter(d['plan_type'] for g in graphs.values() for _,_,d in g.edges(data=True))),node_dataset_types=dict(Counter(d['dataset_type'] for g in graphs.values() for _,d in g.nodes(data=True))),edge_dataset_types=dict(Counter(d['dataset_type'] for g in graphs.values() for _,_,d in g.edges(data=True))),missing_mappings=dict(nodes=missing_assets,edges=missing_edges),orphan_edges=0,parallel_scenarios=sorted({sid for (sid,u,v),n in pairs.items() if n>1}),parallel_pairs=sum(n>1 for n in pairs.values()),graph_class='MultiDiGraph',node_mapping=ASSET_PLAN_TYPE,edge_mapping=REL_PLAN_TYPE,intervention_linked_edges=sum(v['intervention_linked_edges'] for v in per_scenario.values()),shared_groups=sum(len(v['shared_groups']) for v in per_scenario.values()))
save_stage('01_potential',graph_pack(graphs))
report('01',metrics)


PASS: 00_recon current input fingerprints
PASS: 00_recon current source fingerprint
PASS: 00_recon current execution run
PASS: Every dataset type has a mapping
PASS: All original node IDs and dataset types preserved
PASS: All original edge IDs, endpoints and dataset types preserved
PASS: Node counts equal input rows
PASS: Edge counts equal input rows
PASS: Entry points and critical targets exist


PASS: Every endpoint refers to an input asset
{
  "scenarios": 60,
  "nodes": 1576,
  "edges": 3589,
  "per_scenario": {
    "scn_0001": {
      "nodes": 35,
      "edges": 52,
      "entry_points": [
        "external_entry_001",
        "external_entry_002"
      ],
      "critical_targets": [
        "db_003"
      ],
      "shared_groups": {
        "identity_001": [
          "edge_002",
          "edge_014",
          "edge_037"
        ],
        "identity_002": [
          "edge_019",
          "edge_036",
          "edge_038"
        ],
        "identity_003": [
          "edge_023",
          "edge_027",
          "edge_048"
        ],
        "identity_004": [
          "edge_045"
        ],
        "identity_005": [
          "edge_039"
        ],
        "identity_admin_001": [
          "edge_012"
        ],
        "identity_admin_002": [
          "edge_007"
        ],
        "identity_admin_003": [
          "edge_024",
          "edge_043"
        ],
        "cloud_i